# Explore Results for a Specific Run

This notebook loads and explores results from a specific environment/method/num_episodes combination.

For example: all batches for DQN on CartPole with 50 episodes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Configuration

Set the experiment parameters you want to explore:

In [ ]:
# Configuration - modify these to explore different runs
experiment_id = "test_run"
method = "dqn"  # e.g., "monte_carlo", "dqn", "least_squares_mc", "least_squares_td"
num_episodes = 50  # number of episodes used for training

# Paths
base_path = Path("../experiments") / experiment_id
results_path = base_path / "results" / method / str(num_episodes)
predictions_file = results_path / "predictions.parquet"
metadata_file = results_path / "predictions_metadata.json"

print(f"Loading results from: {results_path}")
print(f"Predictions file exists: {predictions_file.exists()}")
print(f"Metadata file exists: {metadata_file.exists()}")

## Load Data

In [ ]:
# Load predictions
df = pd.read_parquet(predictions_file)
print(f"Loaded {len(df)} predictions")
print(f"\nColumns: {list(df.columns)}")
print(f"\nShape: {df.shape}")
df.head()

In [ ]:
# Load metadata
with open(metadata_file, 'r') as f:
    metadata = json.load(f)

print("Metadata:")
for key, value in metadata.items():
    if key not in ['estimator_config', 'network_config']:
        print(f"  {key}: {value}")

print("\nEstimator Config:")
for key, value in metadata['estimator_config'].items():
    print(f"  {key}: {value}")

print("\nNetwork Config:")
for key, value in metadata['network_config'].items():
    print(f"  {key}: {value}")

## Basic Statistics

In [ ]:
# Summary statistics
print("Summary Statistics:")
print(df.describe())

In [ ]:
# Compute prediction errors
df['prediction_error'] = df['predicted_value'] - df['true_value']
df['absolute_error'] = df['prediction_error'].abs()
df['squared_error'] = df['prediction_error'] ** 2

print(f"Mean Squared Error: {df['squared_error'].mean():.4f}")
print(f"Mean Absolute Error: {df['absolute_error'].mean():.4f}")
print(f"Mean Prediction Error (bias): {df['prediction_error'].mean():.4f}")
print(f"Std Dev of Prediction Error: {df['prediction_error'].std():.4f}")

## Analysis by Batch

In [ ]:
# Statistics per batch
batch_stats = df.groupby('batch_idx').agg({
    'predicted_value': ['mean', 'std'],
    'true_value': ['mean', 'std'],
    'squared_error': 'mean',
    'absolute_error': 'mean',
    'prediction_error': ['mean', 'std']
}).round(4)

batch_stats.columns = ['_'.join(col).strip() for col in batch_stats.columns.values]
print("\nStatistics by Batch:")
batch_stats

## Visualizations

In [ ]:
# Plot 1: Predicted vs True Values (scatter plot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall scatter
axes[0].scatter(df['true_value'], df['predicted_value'], alpha=0.3, s=10)
axes[0].plot([df['true_value'].min(), df['true_value'].max()], 
             [df['true_value'].min(), df['true_value'].max()], 
             'r--', label='Perfect prediction')
axes[0].set_xlabel('True Value')
axes[0].set_ylabel('Predicted Value')
axes[0].set_title('Predicted vs True Values (All Batches)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# By batch
for batch_idx in df['batch_idx'].unique():
    batch_df = df[df['batch_idx'] == batch_idx]
    axes[1].scatter(batch_df['true_value'], batch_df['predicted_value'], 
                   alpha=0.5, s=10, label=f'Batch {batch_idx}')
axes[1].plot([df['true_value'].min(), df['true_value'].max()], 
            [df['true_value'].min(), df['true_value'].max()], 
            'k--', label='Perfect prediction', linewidth=2)
axes[1].set_xlabel('True Value')
axes[1].set_ylabel('Predicted Value')
axes[1].set_title('Predicted vs True Values (By Batch)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Distribution of prediction errors
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall histogram
axes[0].hist(df['prediction_error'], bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(0, color='r', linestyle='--', linewidth=2, label='Zero error')
axes[0].axvline(df['prediction_error'].mean(), color='g', linestyle='--', 
               linewidth=2, label=f'Mean error: {df["prediction_error"].mean():.3f}')
axes[0].set_xlabel('Prediction Error')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Prediction Errors')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# By batch (boxplot)
df.boxplot(column='prediction_error', by='batch_idx', ax=axes[1])
axes[1].axhline(0, color='r', linestyle='--', linewidth=2, label='Zero error')
axes[1].set_xlabel('Batch Index')
axes[1].set_ylabel('Prediction Error')
axes[1].set_title('Prediction Error Distribution by Batch')
axes[1].legend()
plt.suptitle('')  # Remove the default boxplot title

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Error metrics by batch
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

batch_indices = sorted(df['batch_idx'].unique())

# MSE by batch
mse_by_batch = [df[df['batch_idx'] == idx]['squared_error'].mean() for idx in batch_indices]
axes[0].bar(batch_indices, mse_by_batch, alpha=0.7, edgecolor='black')
axes[0].axhline(np.mean(mse_by_batch), color='r', linestyle='--', 
               linewidth=2, label=f'Mean MSE: {np.mean(mse_by_batch):.4f}')
axes[0].set_xlabel('Batch Index')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('MSE by Batch')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# MAE by batch
mae_by_batch = [df[df['batch_idx'] == idx]['absolute_error'].mean() for idx in batch_indices]
axes[1].bar(batch_indices, mae_by_batch, alpha=0.7, edgecolor='black', color='orange')
axes[1].axhline(np.mean(mae_by_batch), color='r', linestyle='--', 
               linewidth=2, label=f'Mean MAE: {np.mean(mae_by_batch):.4f}')
axes[1].set_xlabel('Batch Index')
axes[1].set_ylabel('Mean Absolute Error')
axes[1].set_title('MAE by Batch')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Value distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of true values
axes[0].hist(df['true_value'], bins=50, alpha=0.7, label='True values', edgecolor='black')
axes[0].axvline(df['true_value'].mean(), color='r', linestyle='--', 
               linewidth=2, label=f'Mean: {df["true_value"].mean():.3f}')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of True Values')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution of predicted values
axes[1].hist(df['predicted_value'], bins=50, alpha=0.7, label='Predicted values', 
            edgecolor='black', color='orange')
axes[1].axvline(df['predicted_value'].mean(), color='r', linestyle='--', 
               linewidth=2, label=f'Mean: {df["predicted_value"].mean():.3f}')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Predicted Values')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Episode-Level Analysis

In [ ]:
# Statistics per episode
episode_stats = df.groupby(['batch_idx', 'episode_idx']).agg({
    'predicted_value': ['mean', 'std'],
    'true_value': ['mean', 'std'],
    'squared_error': 'mean',
    'absolute_error': 'mean',
    'state_idx': 'count'  # number of states in episode
}).round(4)

episode_stats.columns = ['_'.join(col).strip() for col in episode_stats.columns.values]
episode_stats.rename(columns={'state_idx_count': 'n_states'}, inplace=True)

print("\nEpisode Statistics (first 10 episodes):")
episode_stats.head(10)

In [ ]:
# Plot: MSE vs episode length
fig, ax = plt.subplots(figsize=(10, 6))

for batch_idx in df['batch_idx'].unique():
    batch_episode_stats = episode_stats.loc[batch_idx]
    ax.scatter(batch_episode_stats['n_states'], 
              batch_episode_stats['squared_error_mean'],
              alpha=0.6, s=50, label=f'Batch {batch_idx}')

ax.set_xlabel('Episode Length (number of states)')
ax.set_ylabel('Mean Squared Error')
ax.set_title('MSE vs Episode Length')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Custom Analysis

Use the cells below for your own exploratory analysis:

In [ ]:
# Your custom analysis here
